# RQ1: add input-stage IH to the existing E / zero-one panel

**Current next run:** six-question input IH calibration, then the gated **30-question × 576-position** sweep (GQA 12, MMStar 12, TextVQA 6). Original E/zero records are reused only after strict feature, reference and zero-audit checks. The six-question layer-16 IH sweep is already complete; it does not need rerunning.

Push the new code, import this notebook as an **editable draft**, enable **Internet + T4 ×2**, and attach **`research/results/rq1-ih-inputs-v1.zip`** as a private dataset or upload. This new 34.1 MiB bundle contains the old images/manifests plus the exact historical E/zero records. The old `rq1-inputs-v1.zip` alone is insufficient. Kaggle may unpack the ZIP; either form is detected automatically. No dataset needs to be public.

Set only `RUN_INPUT_IH_CALIBRATION = True` first; leave the full flag and historical flags False. Run cells in order. See `research/RQ1_INPUT_IH.md`. The historical zeroing cells below remain available but are disabled. Default flags prevent accidental work during Kaggle import.


In [ ]:
import json
import os
import sys
import shutil
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_INPUT_IH_CALIBRATION = False  # Current next run: set True. Six cases x 32 positions.
RUN_FULL_INPUT_IH = False  # Keep False until calibration is reviewed.
IH_CALIBRATION_REPORT = None  # Only for resuming with an existing passed input-IH report.json.

RUN_TARGET_AUDIT = False  # Next after the first calibration: diagnose answer casing/token IDs.
RUN_CALIBRATION = False  # Set True for the first two-question GQA run.
RUN_GQA_PILOT = False    # Later: True after calibration controls/timing look sensible.
RUN_TEXTVQA = False      # Optional: all 100 questions, baseline + input-only generation.
RUN_MMSTAR_CHECK = False # Later: 3 questions x 32 tokens, after GQA measurement validation.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None  # Optional exact ZIP or extracted-folder path; new IH bundle: rq1-ih-inputs-v1.zip
CHECKPOINT_ARGS = []  # Optional: ["--model", "/kaggle/input/...", "--vision-model", "/kaggle/input/..."]

if not any([RUN_INPUT_IH_CALIBRATION, RUN_FULL_INPUT_IH, RUN_TARGET_AUDIT, RUN_CALIBRATION, RUN_GQA_PILOT, RUN_TEXTVQA, RUN_MMSTAR_CHECK]):
    raise RuntimeError("Set RUN_INPUT_IH_CALIBRATION = True for the next run, then execute cells manually.")
print("Preparing repository for RQ1 input IH...", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
original_python = repo / "environments/original/.venv/bin/python"
inputs = Path("/kaggle/working/rq1-inputs-v1")
print("Repository:", repo, flush=True)


## Prepare inputs and the original locked environment

The notebook kernel stays unchanged. The setup checks input integrity, installs/reuses the original isolated runtime, and runs tiny native tests before loading pretrained weights. Kaggle auto-unpacked input datasets are supported. The new IH test copies data into a temporary test folder, so the attached private dataset stays read-only.


In [ ]:
if RUN_INPUT_IH_CALIBRATION or RUN_FULL_INPUT_IH:
    sys.path.insert(0, str(repo / "scripts"))
    from rq1_input_ih import locate_inputs
    inputs = locate_inputs(
        [Path("/kaggle/input"), Path("/kaggle/working")],
        Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
    )
else:
    if not (inputs / "protocol.json").exists():
        input_zip = Path(INPUT_ZIP) if INPUT_ZIP else None
        if input_zip is None:
            zip_candidates = list(Path("/kaggle/input").rglob("rq1-inputs-v1.zip"))
            zip_candidates += list(Path("/kaggle/working").glob("rq1-inputs-v1.zip"))
            extracted = []
            for path in Path("/kaggle/input").rglob("protocol.json"):
                try:
                    if json.loads(path.read_text()).get("version") == "rq1-input-zero-v1":
                        if all((path.parent / name / "manifest.json").is_file() for name in ["gqa", "mmstar", "textvqa"]):
                            extracted.append(path.parent)
                except (ValueError, OSError):
                    pass
            if len(zip_candidates) == 1:
                input_zip = zip_candidates[0]
            elif len(extracted) == 1:
                inputs = extracted[0]  # Kaggle may unpack a ZIP uploaded as a Dataset.
            else:
                raise RuntimeError(f"Attach rq1-inputs-v1.zip or its extracted files, and set INPUT_ZIP if needed. ZIPs: {zip_candidates}; folders: {extracted}")
        if input_zip is not None and input_zip.is_dir():
            inputs = input_zip
        elif input_zip is not None:
            inputs.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(input_zip) as z:
                for item in z.infolist():
                    destination = (inputs / item.filename).resolve()
                    if inputs.resolve() not in destination.parents:
                        raise ValueError("Unexpected ZIP path")
                z.extractall(inputs)
print("Accepted input directory:", inputs)
print(json.loads((inputs / "protocol.json").read_text()))
subprocess.run([
    "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
    "--output", "/kaggle/working/rq1-runtime.json",
], cwd=repo, check=True)
# Explicit import prevents Torch-dependent tests silently skipping on Kaggle.
subprocess.run([str(original_python), "-X", "utf8", "-I", "-c", "import torch; print(torch.__version__)"], check=True)
for test in ["research/tests/test_rq1_input.py", "research/tests/test_rq1_native.py"]:
    subprocess.run([str(original_python), "-X", "utf8", "-I", test], cwd=repo, check=True)

if RUN_INPUT_IH_CALIBRATION or RUN_FULL_INPUT_IH:
    subprocess.run(
        [str(original_python), "-X", "utf8", "-I", "research/tests/test_rq1_input_ih.py"],
        cwd=repo, check=True, env=dict(os.environ, RQ1_IH_TEST_INPUTS=str(inputs)),
    )
    subprocess.run(
        [str(original_python), "-X", "utf8", "-I", "research/tests/test_rq1_input_ih_native.py"],
        cwd=repo, check=True,
    )


## New: input IH calibration — run this first

Six fixed cases: the first two recorded GQA, MMStar and TextVQA questions, each at the original first 32 positions. This adds **192 retain-one measurements** and **18 zeroing audit repeats**, plus fresh baseline/no-op/null/final controls. Each case must reproduce the archived E, features and reference/zero scores exactly before its IH loop starts. The new operator physically removes other visual embeddings **before block 1**, preserving all text embeddings and original position gaps.

Download calibration ZIP **and log** for review. This calibration checks measurement/consistency and estimates runtime; it does not establish that a sample maximum represents all 576 positions.


In [ ]:
def run_input_ih(tokens=32, calibration=None):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    label = "rq1-input-ih-calibration-v1" if tokens == 32 else "rq1-input-ih-thirtycases-v1"
    folder = Path(f"/kaggle/working/{label}-{stamp}")
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq1_input_ih.py",
               "--inputs", str(inputs), "--output", str(folder), "--tokens", str(tokens),
               "--download-budget-gib", "20", *CHECKPOINT_ARGS]
    if tokens == 576:
        if calibration is None:
            raise ValueError("Supply the passed input-IH calibration directory/report; the layer-16 report is incompatible.")
        report_path = Path(calibration)
        if report_path.is_dir():
            report_path = report_path / "report.json"
        command += ["--calibration-report", str(report_path)]
    print("Run directory:", folder, flush=True)
    log = folder.with_suffix(".log")
    with log.open("w", encoding="utf-8") as stream:
        process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
    if (folder / "report.json").exists():
        report = json.loads((folder / "report.json").read_text(encoding="utf-8"))
        print("Status:", report["status"], "wall seconds:", round(report["wall_seconds"], 1))
        for question in report.get("questions", []):
            print(question)
    print("Download:", folder.with_suffix(".zip"), "and", log)
    if status:
        raise RuntimeError("Run failed; download its ZIP/log. Do not relax reuse checks or retarget answers.")
    return folder


In [ ]:
input_ih_calibration_dir = None
input_ih_full_dir = None
if RUN_INPUT_IH_CALIBRATION:
    input_ih_calibration_dir = run_input_ih()


## Later: all 30 input questions × 576 positions

Leave `RUN_FULL_INPUT_IH = False` for the first run. After the calibration is reviewed, add/run this one cell in the same live notebook:

```python
input_ih_full_dir = run_input_ih(tokens=576, calibration=input_ih_calibration_dir)
```

No setup rerun or model/checkpoint download is needed in that live session; the helper loads the model once for each run. The full run adds **17,280 IH measurements** and 90 zeroing audit repeats, joins the original E/zero scores, and exactly checks the calibration's 192 IH repeats. It preserves the full 30-case cohort, including the existing MMStar quality/grounding concerns; these are not silently excluded.

Alternatively set the full flag and an existing compatible `IH_CALIBRATION_REPORT` path. Changing input/native/measurement sources requires a new calibration. Do not update Git during an active run.


In [ ]:
if RUN_FULL_INPUT_IH:
    prerequisite = IH_CALIBRATION_REPORT or input_ih_calibration_dir
    input_ih_full_dir = run_input_ih(tokens=576, calibration=prerequisite)


In [ ]:
from IPython.display import FileLink, display
for folder in [input_ih_calibration_dir, input_ih_full_dir]:
    if folder is not None:
        display(FileLink(str(folder.with_suffix(".zip"))))
        display(FileLink(str(folder.with_suffix(".log"))))


## Historical zeroing/generation cells — disabled by default

The following cells reproduce earlier work. Keep their flags False for the new IH experiment; their scores are already bundled as reference evidence. The new run does not require rerunning the old target audits, TextVQA screening or full zeroing sweeps.


## First run: GQA calibration (2 questions on one image × 32 tokens)

The two questions share exact image features, native ApET reconstruction errors and sampled token indices. Controls check repeated reference scores, a clone-only no-op, cached/native forward agreement and a final reference. A failure stops the run and saves a ZIP. Download it even when a subprocess raises an error.

The runner saves reference-answer probabilities rather than asking the model to generate an answer 576 times. Positive sensitivity means the reference becomes less probable after zeroing a patch embedding; it is not an accuracy result.


In [ ]:
def run_experiment(label, manifest, options):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/{label}-{stamp}")
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq1_input.py",
               "--manifest", str(inputs / manifest), "--output", str(folder),
               "--download-budget-gib", "20", *CHECKPOINT_ARGS, *options]
    print("Run directory:", folder, flush=True)
    log = folder.with_suffix(".log")
    with log.open("w") as stream:
        process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if (folder / "report.json").exists():
        report = json.loads((folder / "report.json").read_text())
        print("Status:", report["status"], "wall seconds:", round(report["wall_seconds"], 1))
        for question in report.get("questions", []):
            print(question)
    print("Download:", folder.with_suffix(".zip"), "and", log)
    if status:
        raise RuntimeError(f"Run failed (exit {status}); download the report ZIP/log before restarting.")
    return folder

calibration_dir = None
if RUN_CALIBRATION:
    calibration_dir = run_experiment("rq1-gqa-calibration-v2", "gqa/manifest.json",
                                   ["--questions", "2", "--tokens", "32"])


## Optional target-format diagnostics (already completed for the first pair)

The audit confirmed that lowercase `no`/`stove` were poor reference spellings for this model's capitalized answers. Measurement v2 fixes the GQA target's initial-capital rendering before generation and adds generated-prefix/token-score alignment checks. **The next run is the small v2 calibration**, with `RUN_CALIBRATION=True`, `RUN_TARGET_AUDIT=False`, and other flags False. Keep the full pilot off until v2 is inspected.

This separate audit cell remains available if another format mismatch appears. It records raw/capitalized target scores and actual generation tokens without interventions; generated candidates are diagnostics, never gold labels.


In [ ]:
if RUN_TARGET_AUDIT:
    target_audit_dir = run_experiment("rq1-gqa-target-audit", "gqa/manifest.json",
                                     ["--mode", "target-audit", "--questions", "2"])


## Inspect calibration before expanding

Download the calibration ZIP and open `analysis/index.html` after extracting it locally. It contains the actual 336×336 CLIP canvas, native error map, sensitivity map and per-question scatter plot. Gray patches were not measured. `analysis/tokens.jsonl` preserves all 576 indices with `null` for unmeasured sensitivity.

A passed execution is a measurement check, not an established RQ1 finding. Send the calibration ZIP first. The extrapolation below includes repeated controls but excludes checkpoint loading and is only a rough estimate. Keep the full-pilot flag off until the measurement looks sensible.


In [ ]:
if calibration_dir is not None:
    report = json.loads((calibration_dir / "report.json").read_text())
    for row in report.get("questions", []):
        print(row["case_id"], "rho:", row["spearman"], "controls/status:", row["status"])
    total = sum(row["seconds"] for row in report.get("questions", []))
    if total:
        print("Rough 12 x 576 estimate (minutes):", round(total / (2 * 32) * (12 * 576) / 60, 1))
    from IPython.display import FileLink
    display(FileLink(str(calibration_dir.with_suffix(".zip"))))


## Optional next runs in the same live notebook

Each command loads the model once for its run and releases it when the subprocess exits. The environment and checkpoint downloads are reused. Enable only the run you want. Do not update repository code halfway through a running subprocess.

- GQA: 12 frozen questions on six images, including calibration pair, all 576 patches.
- TextVQA: all 100 retained questions; baseline and input-only 288 generation, native ten-answer consensus scoring. No interventions, OCR transcripts or new filtering.
- MMStar: optional three-question measurement check after GQA validation; full reviewed selection remains packaged for later expansion.


In [ ]:
if RUN_GQA_PILOT:
    gqa_dir = run_experiment("rq1-gqa-input-pilot", "gqa/manifest.json",
                             ["--questions", "12", "--tokens", "576"])


In [ ]:
if RUN_TEXTVQA:
    textvqa_dir = run_experiment("rq1-textvqa-baseline", "textvqa/manifest.json",
        ["--mode", "textvqa", "--configurations", "baseline", "input-only"])


In [ ]:
if RUN_MMSTAR_CHECK:
    mmstar_dir = run_experiment("rq1-mmstar-calibration", "mmstar/manifest.json",
                                ["--questions", "3", "--tokens", "32"])


## Download before stopping the session

In the Output panel download each `rq1-*.zip` and its `.log`. Keep the original ZIPs under `research/results/` locally. A ZIP includes predictions or token measurements, manifests, used images, source snapshots and the exact original `uv.lock`. No model weights or environment directories are bundled.

To rerun, use a new timestamped output directory. This small pilot has token checkpoints for inspection, but does not implement resume; changing controls or measurement policy should create a new run.
